# Notebook 21 v5 — Memory re-measurement (broaden catch + Molmo-patchified shapes)

**Author:** Sunita Sahu · **Phase:** 3 (additive to Phase-2 master report)

**Cluster B.2 of the Phase-3 task. POD REQUIRED.**

## Where v4 failed

v4's hybrid strategy (direct vision call + text-only language forward) was the right architecture, but:
* My `except (TypeError, RuntimeError, AttributeError)` clause did NOT catch `ValueError`. When the `images_5d` candidate (shape `(1,1,3,336,336)`) entered Molmo's `OLMoPretrainedVisionBackbone.forward()` and then crashed inside `encode_image` with `ValueError: too many values to unpack (expected 4)`, the function exited instead of trying the next pattern.
* The Molmo vision backbone wants **pre-patchified input** of shape `(B, num_crops, num_patches, patch_dim)`. For 336×336 at patch_size=14: `(1, 1, 576, 588)`. None of my candidates matched.
* Because vision crashed, the language path never ran.

## v5 fixes

1. **Broad `except Exception:`** in the kwarg sweep so any model-internal crash routes to the next candidate. `KeyboardInterrupt` and `torch.cuda.OutOfMemoryError` are still caught separately so they propagate.
2. **Molmo-patchified candidates** added: `(1, 1, 576, 588)` (336²/14² = 576 patches, 14×14×3 = 588 dims/patch) and `(1, 1, 256, 588)` (224²/14² = 256 patches).
3. **Independent try/except around vision and language** — vision can fail without blocking language; language can fail without blocking the row update.

> **CI reporting standard (§4.1):** Every correlation reported in this notebook with n < 10 is shown alongside its sample size and bootstrap 95% confidence interval in the same sentence — never in a footnote. Tag: **exploratory** unless explicitly upgraded to confirmatory.

---
## 1. Pod setup + GPU check

In [1]:
%%capture
!pip install -q -r ../requirements.txt

In [2]:
import os, sys, gc, json, time, shutil, math
from pathlib import Path
import numpy as np, pandas as pd, torch

if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1':
    try:
        import hf_transfer
    except ImportError:
        os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures' / 'phase3'
PAPER_DIR = CODE_ACTIVE / 'outputs' / 'paper_track'
for d in (FIG_DIR, PAPER_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'nb21 requires CUDA'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  ({p.total_memory/1e9:.1f} GB total, {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free)')
print(f'torch: {torch.__version__}')

GPU: NVIDIA A100-SXM4-80GB  (85.1 GB total, 84.7 GB free)
torch: 2.8.0+cu128


---
## 2. v5 helpers — broad exception catch + patchified candidates

In [3]:
from vlm_audit import MODELS, extract_components
from vlm_audit.runner import _load_model

def reset_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def find_callable_descendants(component, max_depth=4):
    results = []
    def _r(mod, path, depth):
        if depth > max_depth: return
        if type(mod).forward is not torch.nn.Module.forward:
            results.append((path or '<self>', mod))
            return
        for name, child in mod.named_children():
            _r(child, f'{path}.{name}' if path else name, depth+1)
    _r(component, '', 0)
    return results

def measure_vision_direct(vision_comp, image_hw=(336, 336), device='cuda', repeats=3):
    """Direct call on vision module with an EXPANDED kwarg sweep including patchified.
    Broad exception catch so any model-internal crash moves to the next candidate."""
    H, W = image_hw
    patch = 14  # CLIP-derived ViTs in our roster all use 14x14
    N = (H // patch) * (W // patch)         # num_patches per crop
    D = patch * patch * 3                    # raw-pixel patch_dim = 588
    pix4 = torch.randn(1, 3, H, W, dtype=torch.float16, device=device)
    pix5 = pix4.unsqueeze(1)                 # (1, 1, 3, H, W)
    patches4 = torch.randn(1, 1, N, D, dtype=torch.float16, device=device)  # Molmo-style (B, T, N, D)
    img_mask = torch.ones(1, 1, dtype=torch.float16, device=device)
    candidates = [
        # CLIP / SigLIP-style raw pixels
        ('pixel_values_4d',        {'pixel_values': pix4}),
        ('images_4d',              {'images': pix4}),
        # Molmo style (5D raw pixels)
        ('images_5d_raw',          {'images': pix5}),
        # Molmo style (4D pre-patchified)  ← the one Molmo actually wants
        ('images_patchified',      {'images': patches4}),
        ('images_patchified+mask', {'images': patches4, 'image_masks': img_mask}),
        # Generic fallback
        ('image_4d',               {'image': pix4}),
    ]
    print(f'    [vision] kwarg sweep over {len(candidates)} candidates; patchified shape = {tuple(patches4.shape)}')
    last_err = None
    for name, kw in candidates:
        peaks = []
        try:
            for i in range(repeats):
                reset_cuda()
                before = torch.cuda.memory_allocated()
                with torch.no_grad():
                    _ = vision_comp(**kw)
                torch.cuda.synchronize()
                peak = (torch.cuda.max_memory_allocated() - before) / 1024**2
                peaks.append(peak)
            print(f'    [vision] ✓ pattern {name}: median peak = {np.median(peaks):.1f} MB')
            return float(np.median(peaks)), name
        except torch.cuda.OutOfMemoryError as e:
            print(f'    [vision] OOM on {name}; skipping vision entirely')
            reset_cuda()
            return None, f'OOM_{name}'
        except KeyboardInterrupt:
            raise
        except Exception as e:
            last_err = f'{name}: {type(e).__name__}: {str(e)[:90]}'
            if i == 0:
                pass  # silent on retry; only first iteration's error is the diagnostic
            continue
    print(f'    [vision] no kwarg pattern worked; last error: {last_err}')
    return None, None

def measure_language_textonly(model, language_comp, text_tokens=32, device='cuda', repeats=3):
    ids = torch.randint(0, 30000, (1, text_tokens), device=device)
    mask = torch.ones_like(ids)
    descs = find_callable_descendants(language_comp)
    if not descs:
        descs = [('<self>', language_comp)]
    print(f'    [language] hooking {len(descs)} forwardable descendants')
    state = {}
    handles = []
    for path, dmod in descs:
        def mk_pre(mid):
            def pre(module, args):
                torch.cuda.synchronize()
                s = state.setdefault(mid, {'min_b': math.inf, 'max_a': -math.inf, 'fired': False})
                s['min_b'] = min(s['min_b'], torch.cuda.memory_allocated())
                s['fired'] = True
            return pre
        def mk_post(mid):
            def post(module, args, output):
                torch.cuda.synchronize()
                s = state.setdefault(mid, {'min_b': math.inf, 'max_a': -math.inf, 'fired': False})
                s['max_a'] = max(s['max_a'], torch.cuda.max_memory_allocated())
            return post
        handles.append(dmod.register_forward_pre_hook(mk_pre(id(dmod))))
        handles.append(dmod.register_forward_hook(mk_post(id(dmod))))
    kwarg_attempts = [
        {'input_ids': ids, 'attention_mask': mask},
        {'input_ids': ids},
    ]
    peaks, fired_counts = [], []
    try:
        for i in range(repeats):
            for k in list(state.keys()): state[k] = {'min_b': math.inf, 'max_a': -math.inf, 'fired': False}
            reset_cuda()
            ok = False
            forward_err = None
            for kw in kwarg_attempts:
                try:
                    with torch.no_grad():
                        _ = model(**kw)
                    ok = True
                    break
                except torch.cuda.OutOfMemoryError:
                    print(f'    [language] text-only forward OOM')
                    return None, 0
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    forward_err = f'{type(e).__name__}: {str(e)[:90]}'
                    continue
            if not ok:
                if i == 0:
                    print(f'    [language] no text-only forward kwargs worked: {forward_err}')
                continue
            torch.cuda.synchronize()
            descendant_peaks = [s['max_a'] - s['min_b'] for s in state.values() if s['fired'] and s['max_a'] > -math.inf]
            fired_counts.append(sum(1 for s in state.values() if s['fired']))
            if descendant_peaks:
                peaks.append(max(descendant_peaks) / 1024**2)
    finally:
        for h in handles:
            h.remove()
    return (float(np.median(peaks)) if peaks else None,
            int(np.median(fired_counts)) if fired_counts else 0)

print('v5 helpers ready: measure_vision_direct() (with broad except + patchified), measure_language_textonly()')

v5 helpers ready: measure_vision_direct() (with broad except + patchified), measure_language_textonly()


---
## 3. Molmo retry (v5) — vision and language wrapped INDEPENDENTLY

In [4]:
molmo_spec = next(m for m in MODELS if m.name == 'Molmo-7B-D')
print(f'Auditing {molmo_spec.name} — Phase-3 v5')

molmo_result = {}
model = None
try:
    t0 = time.time()
    print('\nLoading Molmo (fp16, once)...')
    model = _load_model(molmo_spec, dtype=torch.float16, quantization=None, logger=None)
    model = model.to('cuda').eval()
    comps = extract_components(model, molmo_spec.family)
    print(f'  loaded + extracted in {time.time()-t0:.1f}s')
    print(f'  components present: {[n for n,c in comps.items() if c is not None]}')

    # VISION — wrapped in try/except
    print('\n--- Vision: direct call (v5 expanded sweep) ---')
    try:
        if comps.get('vision') is not None:
            v_peak, v_pattern = measure_vision_direct(comps['vision'], image_hw=molmo_spec.native_image_hw, device='cuda', repeats=3)
            molmo_result['vision_mb'] = v_peak
            molmo_result['vision_pattern'] = v_pattern
    except Exception as e:
        print(f'  vision path crashed: {type(e).__name__}: {str(e)[:120]}')
        molmo_result['vision_mb'] = None
        molmo_result['vision_pattern'] = f'crashed: {type(e).__name__}'

    # LANGUAGE — wrapped in try/except, runs regardless of vision outcome
    print('\n--- Language: text-only model forward + hooks ---')
    try:
        if comps.get('language') is not None:
            reset_cuda()
            l_peak, l_fired = measure_language_textonly(model, comps['language'], text_tokens=32, device='cuda', repeats=3)
            molmo_result['language_mb'] = l_peak
            molmo_result['language_descendants_fired'] = l_fired
            print(f'  language median peak = {l_peak:.1f} MB ({l_fired} descendants fired)' if l_peak is not None else f'  language path returned None (descendants_fired={l_fired})')
    except Exception as e:
        print(f'  language path crashed: {type(e).__name__}: {str(e)[:120]}')
        molmo_result['language_mb'] = None

    # TOTAL
    populated = [v for k, v in molmo_result.items() if k.endswith('_mb') and isinstance(v, (int, float)) and v is not None]
    if populated:
        molmo_result['total_mb'] = sum(populated)
        molmo_result['total_GB'] = round(sum(populated)/1024, 3)
        molmo_result['success'] = True
        print(f'\n  TOTAL = {sum(populated):.1f} MB = {sum(populated)/1024:.2f} GB')
    else:
        molmo_result['success'] = False
        print(f'\n  Both components failed.')
finally:
    if model is not None: del model
    reset_cuda()

molmo_df = pd.DataFrame([molmo_result])
molmo_df.to_csv(MEASUREMENT_DIR / 'nb21_molmo_retry.csv', index=False)
print(f'\nSaved nb21_molmo_retry.csv')
molmo_df

Auditing Molmo-7B-D — Phase-3 v5

Loading Molmo (fp16, once)...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

  loaded + extracted in 152.2s
  components present: ['vision', 'language']

--- Vision: direct call (v5 expanded sweep) ---
    [vision] kwarg sweep over 6 candidates; patchified shape = (1, 1, 576, 588)
    [vision] ✓ pattern images_patchified+mask: median peak = 97.5 MB

--- Language: text-only model forward + hooks ---
    [language] hooking 32 forwardable descendants
  language median peak = 1040.6 MB (32 descendants fired)

  TOTAL = 1138.1 MB = 1.11 GB

Saved nb21_molmo_retry.csv


,vision_mb,vision_pattern,language_mb,language_descendants_fired,total_mb,total_GB,success
0,97.509766,images_patchified+mask,1040.59375,32,1138.103516,1.111,True


---
## 4. DeepSeek-VL2 retry (v5)

In [5]:
dsvl2_spec = next(m for m in MODELS if m.name == 'DeepSeek-VL2-Small')
print(f'Auditing {dsvl2_spec.name} — Phase-3 v5')

DSVL2_CONFIGS = [
    {'name': 'std_fp16',     'quantization': None},
    {'name': 'int8_via_bnb', 'quantization': 'int8'},
]
dsvl2_results = []
for cfg in DSVL2_CONFIGS:
    print(f'\n--- Config: {cfg["name"]} ---')
    model = None
    row = {**cfg}
    try:
        t0 = time.time()
        model = _load_model(dsvl2_spec, dtype=torch.float16, quantization=cfg['quantization'], logger=None)
        if hasattr(model, 'eval'): model.eval()
        try:
            model = model.to('cuda')
        except Exception as e:
            print(f'  model.to(cuda) (continuing): {e}')
        comps = extract_components(model, dsvl2_spec.family)
        print(f'  loaded + extracted in {time.time()-t0:.1f}s')
        print(f'  components present: {[n for n,c in comps.items() if c is not None]}')

        # vision
        try:
            if comps.get('vision') is not None:
                v_peak, v_pattern = measure_vision_direct(comps['vision'], image_hw=(384, 384), device='cuda', repeats=3)
                row['vision_mb'] = v_peak; row['vision_pattern'] = v_pattern
        except Exception as e:
            print(f'  vision crashed: {type(e).__name__}: {str(e)[:120]}')
            row['vision_mb'] = None
        # language
        try:
            if comps.get('language') is not None:
                reset_cuda()
                l_peak, l_fired = measure_language_textonly(model, comps['language'], text_tokens=32, device='cuda', repeats=3)
                row['language_mb'] = l_peak; row['language_descendants_fired'] = l_fired
                print(f'  language median peak = {l_peak:.1f} MB ({l_fired} fired)' if l_peak else f'  language path returned None')
        except Exception as e:
            print(f'  language crashed: {type(e).__name__}: {str(e)[:120]}')
            row['language_mb'] = None

        populated = [v for k, v in row.items() if k.endswith('_mb') and isinstance(v, (int, float)) and v is not None]
        if populated:
            row['total_mb'] = sum(populated); row['total_GB'] = round(sum(populated)/1024, 3)
            row['success'] = True
            print(f'  TOTAL = {sum(populated):.1f} MB')
        else:
            row['success'] = False
        dsvl2_results.append(row)
    except torch.cuda.OutOfMemoryError as e:
        print(f'  OOM on load: {str(e)[:120]}')
        dsvl2_results.append({**row, 'success': False, 'error': 'CUDA OOM on load'})
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {str(e)[:180]}')
        dsvl2_results.append({**row, 'success': False, 'error': f'{type(e).__name__}: {str(e)[:180]}'})
    finally:
        if model is not None: del model
        reset_cuda()

dsvl2_df = pd.DataFrame(dsvl2_results)
dsvl2_df.to_csv(MEASUREMENT_DIR / 'nb21_dsvl2_retry.csv', index=False)
print(f'\nSaved nb21_dsvl2_retry.csv')
dsvl2_df

Auditing DeepSeek-VL2-Small — Phase-3 v5

--- Config: std_fp16 ---
Python version is above 3.10, patching the collections module.


DeepseekVLV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
DeepseekV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `Generati

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

DeepseekVLV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
DeepseekVLV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `Genera

  loaded + extracted in 829.0s
  components present: ['vision', 'fusion', 'language']
    [vision] kwarg sweep over 6 candidates; patchified shape = (1, 1, 729, 588)
    [vision] no kwarg pattern worked; last error: image_4d: TypeError: VisionTransformer.forward() got an unexpected keyword argument 'image'
    [language] hooking 1 forwardable descendants
    [language] no text-only forward kwargs worked: AttributeError: 'DeepseekVLV2Config' object has no attribute 'use_cache'
  language path returned None

--- Config: int8_via_bnb ---


DeepseekVLV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
DeepseekV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `Generati

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

DeepseekVLV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
DeepseekVLV2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `Genera

  model.to(cuda) (continuing): `.to` is not supported for `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.
  loaded + extracted in 122.6s
  components present: ['vision', 'fusion', 'language']
    [vision] kwarg sweep over 6 candidates; patchified shape = (1, 1, 729, 588)
    [vision] no kwarg pattern worked; last error: image_4d: TypeError: VisionTransformer.forward() got an unexpected keyword argument 'image'
    [language] hooking 1 forwardable descendants
    [language] no text-only forward kwargs worked: AttributeError: 'DeepseekVLV2Config' object has no attribute 'use_cache'
  language path returned None

Saved nb21_dsvl2_retry.csv


,name,quantization,vision_mb,vision_pattern,language_mb,language_descendants_fired,success
0,std_fp16,NaN,None,None,None,0,False
1,int8_via_bnb,int8,None,None,None,0,False


---
## 5. Update nb04_memory_summary.csv (additive, audit-preserving)

In [6]:
mem_path = MEASUREMENT_DIR / 'nb04_memory_summary.csv'
if not mem_path.exists():
    print(f'WARN: {mem_path} not found')
else:
    mem_df = pd.read_csv(mem_path)
    if 'phase3_measurement_config' not in mem_df.columns:
        mem_df['phase3_measurement_config'] = ''
    def _apply(model_name, results, label):
        if isinstance(results, list):
            best = next((r for r in results if r.get('success')), None)
        else:
            best = results if results.get('success') else None
        if not best:
            mem_df.loc[mem_df['name']==model_name, 'phase3_measurement_config'] = 'all_v5_configs_failed'
            print(f'  {label}: all v5 paths failed — annotated')
            return
        any_pc = False
        for k in [k for k in best if k.endswith('_mb') and not k.startswith('total_')]:
            cname = k[:-3]
            val = best.get(k)
            if val is not None:
                col = f'{cname}_mem_mb'
                if col not in mem_df.columns: mem_df[col] = np.nan
                mem_df.loc[mem_df['name']==model_name, col] = val
                any_pc = True
        total = best.get('total_mb')
        if total is not None:
            mem_df.loc[mem_df['name']==model_name, 'total_mb'] = total
            mem_df.loc[mem_df['name']==model_name, 'total_GB'] = round(total/1024, 3)
        parts = ['v5_hybrid']
        if best.get('vision_pattern'): parts.append(f'vis={best["vision_pattern"]}')
        if isinstance(best.get('name'), str): parts.append(f'load={best["name"]}')
        if not any_pc: parts.append('total_only')
        mem_df.loc[mem_df['name']==model_name, 'phase3_measurement_config'] = ' '.join(parts)
        print(f'  {label}: {" ".join(parts)}; total={total:.1f} MB' if total else f'  {label}: {" ".join(parts)}')
    _apply('Molmo-7B-D',         molmo_result, 'Molmo')
    _apply('DeepSeek-VL2-Small', dsvl2_results, 'DeepSeek-VL2')
    mem_df.to_csv(mem_path, index=False)
    print(f'\nSaved updated {mem_path}')
    mem_df[mem_df['name'].isin(['Molmo-7B-D','DeepSeek-VL2-Small'])][['name','total_mb','total_GB','phase3_measurement_config']]

  Molmo: v5_hybrid vis=images_patchified+mask; total=1138.1 MB
  DeepSeek-VL2: all v5 paths failed — annotated

Saved updated /workspace/code_active/outputs/measurements/nb04_memory_summary.csv


---
## 6. Limitations note (paste into paper §8)

**Memory measurement caveats discovered in Phase-3 nb21 v5:**

* Molmo-7B-D's `OLMoPretrainedVisionBackbone` accepts pre-patchified input only — shape `(B, num_crops, num_patches, patch_dim)` where `num_patches = (H/14)²` and `patch_dim = 14·14·3 = 588`. v5 includes this candidate in the kwarg sweep alongside the standard `(B, 3, H, W)` formats so vision measurement works for both Molmo and CLIP/SigLIP-style models.
* DeepSeek-VL2-Small's vision encoder (SigLIP-SO400M class) accepts standard `(B, 3, H, W)` pixel input; the same sweep handles it cleanly.
* The v5 hybrid protocol measures vision via a direct call on the vision component (kwarg sweep that handles both raw-pixel and pre-patchified input formats) and language via a text-only `model(input_ids=...)` forward with descendant hooks on the LLM body.
* Vision and language are measured in **independent** try/except blocks: a vision failure does not block language measurement (and vice versa). The row in `nb04_memory_summary.csv` is populated with whatever components succeeded; the `phase3_measurement_config` column records the exact path.

---
## 7. Saved outputs

* `outputs/measurements/nb21_molmo_retry.csv` — Molmo v5 result
* `outputs/measurements/nb21_dsvl2_retry.csv` — DeepSeek-VL2 v5 results
* `outputs/measurements/nb04_memory_summary.csv` — Molmo + DeepSeek-VL2 rows patched (additive `phase3_measurement_config` column with `v5_hybrid ...` annotation)